In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table('workspace.default.sales_bronze_capstone')
valid_df = (bronze_df
    .withColumn("order_date", to_date("order_date"))
    .withColumn("customer_id", trim("customer_id"))
    .withColumn("customer_name", trim("customer_name"))
    .withColumn("city", trim("city"))
    .withColumn("state", trim("state"))
    .withColumn("product_id", trim("product_id"))
    .withColumn("product_name", trim("product_name"))
    .withColumn("category", trim("category"))
    .withColumn("payment_method", trim("payment_method"))
    .withColumn("order_status", trim("order_status"))
    .withColumn("unit_price", round("unit_price", 2))
    .withColumn("gross_amount", round("gross_amount", 2))
    .withColumn("discount_amount", round("discount_amount", 2))
    .withColumn("net_amount", round("net_amount", 2))
    .dropDuplicates(["order_id"])
)

silver_df = (
    valid_df.filter(
        (col("order_id").isNotNull()) &
        (col("order_date").isNotNull()) &
        (col("quantity") > 0) &
        (col("unit_price") >= 0) &
        (col("discount_pct").between(0, 100))
    ) 
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .withColumn("month_name", date_format("order_date", "MMMM"))
)

# silver_df.printSchema()

silver_df.write.mode("overwrite").saveAsTable("workspace.default.sales_silver_capstone")

